# 06 · Biomatemática: crescimento logístico e envelhecimento da mortalidade

Duas aplicações de modelos clássicos da biologia matemática aos dados do WPP2024:

1. **Crescimento logístico da população (1950–2023).** Um modelo com capacidade de suporte *K*, taxa intrínseca *r* e ponto de inflexão *t₀*. Ajustamos por país e comparamos a extrapolação com a projeção Medium da ONU.
2. **Envelhecimento da mortalidade (lei de Gompertz).** A mortalidade adulta cresce exponencialmente com a idade, com taxa *b*. A partir das probabilidades de morte por faixa etária, estimamos *b* para cada país.

**Limitações:** o modelo logístico supõe uma única capacidade de suporte, o que é uma simplificação. A estimativa de *b* usa poucas faixas etárias e é uma aproximação, não a tábua de vida completa.

In [1]:
import sys
from pathlib import Path

# Permite importar src/ a partir da pasta notebooks/
if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

import altair as alt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

from src.dados import carregar_medium, paises

RAIZ = Path.cwd().parent
medium = carregar_medium()
paises_medium = paises(medium)
PAISES_EXEMPLO = ["Brazil", "Nigeria", "Japan", "India", "Germany", "Sweden"]

# --- Figuras: fundo branco, texto preto, salvas em figures/ ---
import re as _re
import vl_convert as vlc
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_FIG = RAIZ / "figures"
PASTA_FIG.mkdir(exist_ok=True)
PREFIXO_FIG = "06"
_renderer_padrao = alt.renderers.get()
_contador_fig = {"n": 0}

def _titulo_de(spec):
    t = spec.get("title")
    if isinstance(t, dict):
        t = t.get("text")
    if isinstance(t, list):
        t = " ".join(map(str, t))
    if isinstance(t, str) and t:
        return t
    for chave in ("layer", "hconcat", "vconcat", "concat"):
        for sub in spec.get(chave, []):
            r = _titulo_de(sub)
            if r:
                return r
    return None

def _nome_grafico(spec):
    t = _titulo_de(spec) or "chart"
    slug = _re.sub(r"[^a-zA-Z0-9]+", "_", str(t).lower()).strip("_")
    return slug[:60] or "chart"

def _renderer_com_figuras(spec, **metadata):
    _contador_fig["n"] += 1
    base = f"{PREFIXO_FIG}_{_contador_fig['n']:02d}_{_nome_grafico(spec)}"
    spec_branco = dict(spec, background="white")
    (PASTA_FIG / f"{base}.png").write_bytes(vlc.vegalite_to_png(spec_branco, scale=2))
    if '"params"' in str(spec) and '"bind"' in str(spec):
        alt.VegaLite(spec_branco).save(str(PASTA_FIG / f"{base}.html"))
    return _renderer_padrao(spec, **metadata)

alt.renderers.register("com_figuras", _renderer_com_figuras)
alt.renderers.enable("com_figuras")


RendererRegistry.enable('com_figuras')

## Fundamentos

- **Crescimento logístico (Verhulst, 1838):** o crescimento é proporcional ao tamanho da população e ao espaço ainda disponível. A curva tem três fases: crescimento lento, aceleração até o ponto de inflexão *t₀* e desaceleração perto da capacidade de suporte *K*. É o modelo clássico de populações limitadas por recursos.
- **Lei de Gompertz (1825):** a taxa de mortalidade cresce exponencialmente com a idade adulta. Para humanos, a taxa de envelhecimento *b* costuma ficar em torno de 0,08 a 0,09 por ano.
- **Limitação dos modelos:** populações humanas não seguem uma capacidade de suporte fixa; migração, políticas e tecnologia mudam a trajetória. Por isso os modelos são descritivos, e a comparação com a projeção da ONU é o teste principal.

## 1. Crescimento logístico da população

Modelo: P(t) = K / (1 + exp(−r(t − t₀))), com P em milhões. O ajuste usa 1950–2023 (dados estimados). A extrapolação para 2024–2100 é comparada com a projeção Medium.

Limites usados no ajuste: *K* entre o maior valor observado e 10 vezes esse valor, para evitar soluções sem sentido.

In [2]:
def logistica(t, K, r, t0):
    return K / (1 + np.exp(-r * (t - t0)))

def ajustar_logistica(serie_anos, serie_pop):
    pmax = serie_pop.max()
    p0 = [pmax * 1.5, 0.05, 1990]
    limites = ([pmax, 0.0, 1900], [10 * pmax, 1.0, 2100])
    params, _ = curve_fit(logistica, serie_anos, serie_pop, p0=p0, bounds=limites, maxfev=20000)
    return params

resultados = []
previsoes = []
for loc_id, sub in paises_medium.groupby("LocID"):
    sub = sub.sort_values("Time")
    hist = sub[sub["Time"].between(1950, 2023)]
    proj = sub[sub["Time"].between(2024, 2100)]
    anos_h = hist["Time"].values.astype(float)
    pop_h = hist["TPopulation1July"].values / 1000  # milhões
    try:
        K, r, t0 = ajustar_logistica(anos_h, pop_h)
    except Exception:
        continue
    pred_proj = logistica(proj["Time"].values.astype(float), K, r, t0)
    real_proj = proj["TPopulation1July"].values / 1000
    erro = np.mean(np.abs(pred_proj - real_proj) / real_proj) * 100
    resultados.append({"LocID": loc_id, "Location": sub["Location"].iloc[0], "K (milhões)": K,
                       "r": r, "t0": t0, "erro médio 2024–2100 (%)": erro})
    prev = sub[["Location", "Time", "TPopulation1July"]].copy()
    prev["logistica"] = logistica(prev["Time"].values.astype(float), K, r, t0) * 1000
    previsoes.append(prev)

logis = pd.DataFrame(resultados)
previsoes = pd.concat(previsoes, ignore_index=True)
print(f"Países ajustados: {len(logis)} de {paises_medium['LocID'].nunique()}")
logis["erro médio 2024–2100 (%)"].describe().round(1)

Países ajustados: 237 de 237


count    237.0
mean      35.4
std       38.2
min        1.4
25%       11.9
50%       23.6
75%       43.9
max      319.4
Name: erro médio 2024–2100 (%), dtype: float64

Para os países de exemplo, a curva logística ajustada aos dados estimados (linha) é comparada com a projeção Medium (linha tracejada). Os pontos são os dados observados.

In [3]:
def grafico_logistica(pais):
    d = previsoes[previsoes["Location"] == pais].copy()
    d["millions"] = d["TPopulation1July"] / 1000
    d["logistica_mi"] = d["logistica"] / 1000
    base = alt.Chart(d).encode(x=alt.X("Time:Q", title="Year", axis=alt.Axis(format="d"), scale=alt.Scale(zero=False)))
    obs = base.transform_filter("datum.Time <= 2023").mark_circle(size=28, color="#8a8a8a").encode(
        y=alt.Y("milhões:Q", title="Population (millions)"))
    logi = base.mark_line(strokeWidth=2.5, color="#2a78d6").encode(y="logistica_mi:Q")
    medium_linha = base.transform_filter("datum.Time >= 2023").mark_line(
        strokeWidth=2, strokeDash=[5, 4], color="#eb6834").encode(y="milhões:Q")
    return (obs + logi + medium_linha).properties(width=300, height=200, title=pais)

alt.hconcat(*[grafico_logistica(p) for p in PAISES_EXEMPLO[:3]]) & alt.hconcat(*[grafico_logistica(p) for p in PAISES_EXEMPLO[3:]])

alt.VConcatChart(...)

**Leitura:** o ajuste é razoável para grandes países (Brasil, cerca de 19% de erro médio; Alemanha, cerca de 20%; Suécia, 8%). A mediana do erro entre todos os países é de cerca de 24%, mas há grande dispersão: 48 países têm erro abaixo de 10%, e 52 passam de 50%. Os piores casos são pequenas ilhas, com erros acima de 100%. A Nigéria, com erro de 49%, mostra o limite do modelo: a população ainda acelera, e um único ponto de inflexão não captura isso.

Os valores de *K* (capacidade de suporte) também não devem ser lidos como previsão. A Nigéria aparece com *K* perto de 2,2 bilhões, e a Índia com cerca de 2,3 bilhões: são extrapolações do ajuste, não limites demográficos.

## 2. Envelhecimento da mortalidade (lei de Gompertz)

Na lei de Gompertz, a taxa de mortalidade cresce exponencialmente com a idade, com taxa *b*. Quanto maior *b*, mais rápido a mortalidade sobe com a idade (envelhecimento mais intenso).

As colunas `Q` do arquivo são probabilidades **por mil**; elas são convertidas para fração antes do cálculo. A partir delas, calculamos a sobrevivência em 15, 40 e 60 anos:
- S(40) = 1 − Q0040
- S(60) = 1 − Q0060
- S(15) = S(60) / (1 − Q1560)

A taxa de mortalidade média entre 15–40 e entre 40–60 dá duas estimativas, e a razão entre elas define *b*.

In [4]:
ano_ref = 2023
d = paises_medium[paises_medium["Time"] == ano_ref].copy()
# As colunas Q são probabilidades por mil: convertemos para fração
for c in ["Q5", "Q0040", "Q0060", "Q1560"]:
    d[c] = d[c] / 1000
d["S60"] = 1 - d["Q0060"]
d["S40"] = 1 - d["Q0040"]
d["S15"] = d["S60"] / (1 - d["Q1560"])

d["mu_15_40"] = -np.log(d["S40"] / d["S15"]) / 25
d["mu_40_60"] = -np.log(d["S60"] / d["S40"]) / 20
validos = d[(d["mu_15_40"] > 0) & (d["mu_40_60"] > 0)].copy()
validos["b (envelhecimento)"] = np.log(validos["mu_40_60"] / validos["mu_15_40"]) / (50 - 27.5)

print(f"Países com estimativa válida em {ano_ref}: {len(validos)} de {len(d)}")
validos["b (envelhecimento)"].describe().round(3)

Países com estimativa válida em 2023: 237 de 237


count    237.000
mean       0.068
std        0.018
min        0.013
25%        0.056
50%        0.067
75%        0.079
max        0.171
Name: b (envelhecimento), dtype: float64

Relação entre o envelhecimento da mortalidade (*b*) e a expectativa de vida ao nascer.

**Leitura:** a correlação é **positiva** (Spearman ρ = 0,60). Países com maior expectativa de vida têm *b* mais alto: a mortalidade adulta sobe mais rápido com a idade. Por tercis de expectativa de vida, a mediana de *b* vai de 0,056 (baixa) a 0,071 (média) e 0,079 (alta). Isso indica que, nos países de vida longa, a mortalidade cai mais nas idades jovens e a curva fica mais inclinada na vida adulta.

A estimativa de *b* usa só três faixas etárias (15–40 e 40–60), então os valores absolutos são aproximações. A mediana de 0,067 fica abaixo do valor de cerca de 0,08 a 0,09 citado na literatura para adultos. Use *b* como índice relativo entre países, não como valor exato.

In [5]:
grafico_b = alt.Chart(validos).mark_circle(size=60, opacity=0.75, color="#2a78d6").encode(
    x=alt.X("LEx:Q", title="Life expectancy at birth (years)", scale=alt.Scale(zero=False)),
    y=alt.Y("b (envelhecimento):Q", title="Gompertz b (ageing rate)"),
    tooltip=["Location:N", alt.Tooltip("LEx:Q", format=".1f"), alt.Tooltip("b (envelhecimento):Q", format=".3f")],
).properties(width=560, height=340)
grafico_b

alt.Chart(...)

## 3. Curvas de sobrevivência simplificadas

Com S(5), S(15), S(40) e S(60), obtemos uma curva de sobrevivência em cinco pontos para cada país de exemplo. É uma tábua de vida reduzida, útil para comparar a forma geral das curvas.

In [6]:
def curva_sobrevivencia(pais, ano=ano_ref):
    r_ = d[d["Location"] == pais].iloc[0]
    return pd.DataFrame({
        "idade": [0, 5, 15, 40, 60],
        "sobrevivencia": [1.0, 1 - r_["Q5"], r_["S15"], r_["S40"], r_["S60"]],
        "pais": pais,
    })

curvas = pd.concat([curva_sobrevivencia(p) for p in PAISES_EXEMPLO if p in d["Location"].values], ignore_index=True)

alt.Chart(curvas).mark_line(point=True, strokeWidth=2.5).encode(
    x=alt.X("idade:Q", title="Age (years)"),
    y=alt.Y("sobrevivencia:Q", title="Probability of survival", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("pais:N", title="Country"),
    tooltip=["pais:N", "idade:Q", alt.Tooltip("sobrevivencia:Q", format=".3f")],
).properties(width=560, height=340)

alt.Chart(...)

## 4. Ajuste logístico por região

Somamos a população dos países de cada região (cada país entra uma só vez) e ajustamos a logística à série regional. Agregar costuma reduzir o erro: a variação de países pequenos se compensa dentro da região.

In [7]:
geo = medium[medium["LocTypeName"] == "Geographic region"][["LocID", "Location"]].drop_duplicates()
sub = medium[medium["LocTypeName"] == "Subregion"][["LocID", "ParentID"]].drop_duplicates()
nome_geo = dict(zip(geo["LocID"], geo["Location"]))
sub_para_regiao = {int(l): nome_geo.get(int(p)) for l, p in zip(sub["LocID"], sub["ParentID"])}
sub_para_regiao[918] = "Northern America"  # código usado como pai pelos países da América do Norte

pais_regiao = (
    paises_medium.drop_duplicates("LocID")
    .assign(regiao=lambda d: d["ParentID"].map(lambda p: sub_para_regiao.get(int(p), nome_geo.get(int(p))) if pd.notna(p) else None))
    .set_index("LocID")["regiao"]
)
regional = (
    paises_medium.assign(regiao=paises_medium["LocID"].map(pais_regiao))
    .dropna(subset=["regiao"])
    .groupby(["regiao", "Time"], as_index=False)["TPopulation1July"].sum()
)
regional["millions"] = regional["TPopulation1July"] / 1000

linhas_reg = []
previsoes_reg = []
for reg, sub_reg in regional.groupby("regiao"):
    sub_reg = sub_reg.sort_values("Time")
    h = sub_reg[sub_reg["Time"].between(1950, 2023)]
    f = sub_reg[sub_reg["Time"].between(2024, 2100)]
    K, r, t0 = ajustar_logistica(h["Time"].values.astype(float), h["millions"].values)
    pred = logistica(f["Time"].values.astype(float), K, r, t0)
    erro = np.mean(np.abs(pred - f["millions"].values) / f["millions"].values) * 100
    linhas_reg.append({"região": reg, "K (milhões)": round(K), "r": round(r, 4), "t0": round(t0, 1),
                       "erro médio 2024–2100 (%)": round(erro, 1)})
    prev = sub_reg[["Time", "millions"]].copy()
    prev["logistica"] = logistica(prev["Time"].values.astype(float), K, r, t0)
    prev["região"] = reg
    previsoes_reg.append(prev)

logis_regiao = pd.DataFrame(linhas_reg)
previsoes_reg = pd.concat(previsoes_reg, ignore_index=True)
logis_regiao

,região,K (milhões),r,t0,erro médio 2024–2100 (%)
0,Africa,11072,0.0282,2089.2,25.8
1,Asia,6424,0.0337,1990.3,14.5
2,Europe,756,0.0497,1931.4,13.1
3,Latin America and the Caribbean,823,0.0386,1986.2,11.0
4,Northern America,646,0.0191,2002.7,10.3
5,Oceania,236,0.0190,2097.9,29.4


In [8]:
def grafico_regiao(reg):
    d = previsoes_reg[previsoes_reg["região"] == reg]
    base = alt.Chart(d).encode(x=alt.X("Time:Q", title="Year", axis=alt.Axis(format="d"), scale=alt.Scale(zero=False)))
    obs = base.transform_filter("datum.Time <= 2023").mark_circle(size=24, color="#8a8a8a").encode(
        y=alt.Y("milhões:Q", title="Population (millions)"))
    logi = base.mark_line(strokeWidth=2.5, color="#2a78d6").encode(y="logistica:Q")
    med = base.transform_filter("datum.Time >= 2023").mark_line(strokeWidth=2, strokeDash=[5, 4], color="#eb6834").encode(y="milhões:Q")
    return (obs + logi + med).properties(width=280, height=190, title=reg)

alt.vconcat(*[
    alt.hconcat(*[grafico_regiao(r) for r in REGS])
    for REGS in [sorted(previsoes_reg["região"].unique())[i:i + 3] for i in range(0, 6, 3)]
])

alt.VConcatChart(...)

**Leitura:** Europa, Américas do Norte e América Latina têm erro entre 10% e 13%, e a Ásia, cerca de 15%. África (26%) e Oceânia (29%) têm os maiores erros: são as regiões em que a população ainda acelera na projeção, e a logística com um único ponto de inflexão não acompanha isso. A mediana regional, de cerca de 14%, é menor que a mediana por país (24%).

## 5. Salvar resultados

- `data/processed/logistica_populacao.csv`: parâmetros K, r e t₀ por país, com o erro de extrapolação.
- `data/processed/gompertz_envelhecimento.csv`: estimativa de *b* por país em 2023.

In [9]:
SAIDA_DADOS = RAIZ / "data" / "processed"
SAIDA_DADOS.mkdir(parents=True, exist_ok=True)
logis.round(4).to_csv(SAIDA_DADOS / "logistica_populacao.csv", index=False)
validos[["LocID", "Location", "LEx", "mu_15_40", "mu_40_60", "b (envelhecimento)"]].round(5).to_csv(
    SAIDA_DADOS / "gompertz_envelhecimento.csv", index=False)
print("Salvos em", SAIDA_DADOS)

Salvos em /Users/marcosabilio/Documents/GitHub/DataQuest2026/data/processed
